In [2]:
import pandas as pd

In [3]:
df = pd.read_excel('data.xlsx')
sdf= pd.DataFrame()

In [5]:
# Rename Columns for ease
df.columns = ['time','age','gender','class','breakfast', 'lunch', 'dinner', 'dislikes','allergies','comments']
df.head()

,time,age,gender,class,breakfast,lunch,dinner,dislikes,allergies,comments
0,2025-02-06 08:35:06.187,15-18,Female,100 level,Moi-Moi,jollof rice,Indomie and Egg,Eba,"Peanuts, Groundnut",NaN
1,2025-02-06 08:39:28.901,19-22,Female,400 level,"Yam, jollof rice, Bread, white rice, Egg, plan...","fried rice, Yam, Bread, white rice, Beans, Egg...","Beans, jollof rice, fried rice, Bread, white r...","Fish, Soda drink",Nil,Nil
2,2025-02-06 08:40:47.541,15-18,Female,100 level,"white rice, Spaghetti, plantain, Youghurt, Chi...","Fish, Amala",Indomie and Egg,"Beans, Eba",Soy,No
3,2025-02-06 08:41:03.113,19-22,Female,400 level,"jollof rice, fried rice, Bread, white rice, Sp...","fried rice, Yam, Bread, white rice, Beans, Egg...","Shawarma and chips, jollof rice, fried rice, B...",Fish,Diary and milk,NaN
4,2025-02-06 08:41:03.734,15-18,Female,100 level,Cereal,"Bread,",White Rice,"Beans, Amala",Nil,No


In [8]:
# Step 1: Extract all unique food items
all_foods = set()
def setFood(col):
  for item in col:
    foods = [food.strip() for food in item.split(',')]
    for e in foods:
        if e != "":
            all_foods.add(e.lower())

In [10]:
setFood(df['breakfast'])
setFood(df['lunch'])
setFood(df['dinner'])
setFood(df['dislikes'])
setFood(df['allergies'])

In [12]:
all_foods

{'amala',
 'beans',
 'bread',
 'burgers',
 'cereal',
 'chicken',
 'coffee',
 'coochie',
 'diary and milk',
 'eba',
 'egg',
 'egusi',
 'fish',
 'fried rice',
 'fruit salad',
 'fufu',
 'groundnut',
 'ice cream',
 'indomie and egg',
 'jollof rice',
 'meat',
 'moi-moi',
 'nil',
 'okro',
 'omlette',
 'pap',
 'peanut butter',
 'peanuts',
 'pizza',
 'plantain',
 'potash',
 'potato',
 'pounded yam',
 'salad & fruits',
 'seafood (fish)',
 'semo',
 'sharwama',
 'shawarma and chips',
 'smoothie',
 'snails',
 'soda drink',
 'soy',
 'spaghetti',
 'starch',
 'sushi',
 'tea',
 'toast bread',
 'white rice',
 'yam',
 'youghurt'}

In [14]:
len(all_foods)

50

In [16]:
sdf['meal_time'] = ['breakfast', 'lunch', 'dinner']

In [18]:
sdf.head()

,meal_time
0,breakfast
1,lunch
2,dinner


In [20]:
# Step 3: Create new columns for each food item
for food in all_foods:
    for meal_time in ['breakfast', 'lunch', 'dinner']:
        col_name = f"{food}"
        sdf[col_name] = 0

In [22]:
sdf

,meal_time,diary and milk,shawarma and chips,starch,peanut butter,meat,moi-moi,white rice,sharwama,coochie,...,fruit salad,amala,chicken,bread,soda drink,pounded yam,yam,jollof rice,spaghetti,cereal
0,breakfast,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,lunch,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,dinner,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
# Create a new DataFrame with 0s, same number of rows as df, columns are all food items
food_counts = pd.DataFrame(0, index=df.index, columns=sorted(all_foods))

In [26]:
sdf['nil']

0    0
1    0
2    0
Name: nil, dtype: int64

In [28]:
# Example test input
# test = {
#     'breakfast': ['white rice, tea', 'tea, peanut butter', 'tea'],
#     'lunch': ['white rice, chicken, beans', 'beans, moi-moi', 'white rice, shawarma and chips'],
#     'dinner': ['tea, smoothie', 'chicken, okro', 'moi-moi, seafood (fish), tea']
# }

# Iterate over meals and update sdf
for i, meal in enumerate(['breakfast', 'lunch', 'dinner']):
    for meal_entry in df[meal]:
        items = [i.strip().lower() for i in meal_entry.split(',')]
        for food in items:
            if food in sdf.columns:
                sdf.at[i, food] += 1

In [30]:
sdf

,meal_time,diary and milk,shawarma and chips,starch,peanut butter,meat,moi-moi,white rice,sharwama,coochie,...,fruit salad,amala,chicken,bread,soda drink,pounded yam,yam,jollof rice,spaghetti,cereal
0,breakfast,0,0,0,0,47,34,52,0,0,...,1,0,54,85,7,0,41,55,52,3
1,lunch,0,0,0,0,53,0,61,0,0,...,0,52,66,27,26,65,43,73,57,0
2,dinner,0,46,1,0,43,31,44,0,1,...,0,38,54,27,19,43,1,45,51,0


In [32]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report
import pandas as pd

# Assume sdf is your data, with 'meal_time' and binary food columns
meal_map = {'breakfast': 0, 'lunch': 1, 'dinner': 2}
X = sdf[['meal_time']].replace(meal_map)
y = sdf.drop(columns=['meal_time'])

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Use OneVsRest for multi-label classification
model = OneVsRestClassifier(LogisticRegression())
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=y.columns))


/var/folders/x4/q08y3pwn2l1b1cvjq1gt9n440000gn/T/ipykernel_43436/3734471486.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = sdf[['meal_time']].replace(meal_map)


ValueError: Multioutput target data is not supported with label binarization

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

# sdf = pd.DataFrame({
#     'meal_time': ['breakfast', 'lunch', 'dinner'],
#     'egg': [2, 1, 1],
#     'tea': [1, 0, 1],
#     'rice': [1, 1, 1],
#     'yam': [1, 1, 1],
#     'chicken': [0, 1, 0],
#     'bread': [2, 0, 1],
#     'beans': [0, 2, 0],
#     'fish': [0, 0, 1]
# })

le = LabelEncoder()
sdf['meal_time_encoded'] = le.fit_transform(sdf['meal_time'])

X = sdf[['meal_time_encoded']]
y = sdf.drop(columns=['meal_time', 'meal_time_encoded'])
y_binary = (y > 0).astype(int)

clf = OneVsRestClassifier(LogisticRegression(solver='liblinear'))
clf.fit(X, y_binary)

y_pred = clf.predict(X)

print(classification_report(y_binary, y_pred, target_names=y.columns, zero_division=0))

# Recommend for a new meal_time
new_input = pd.DataFrame({'meal_time_encoded': [le.transform(['lunch'])[0]]})
pred = clf.predict(new_input)
recommended_foods = [food for food, present in zip(y.columns, pred[0]) if present == 1]
print("Recommended foods for lunch:", recommended_foods)


In [ ]:
# Recommend for a new meal_time input
new_input = pd.DataFrame({'meal_time_encoded': [le.transform(['breakfast'])[0]]})

pred = clf.predict(new_input)
recommended_foods = [food for food, present in zip(y.columns, pred[0]) if present == 1]

print("Recommended foods for breakfast:", recommended_foods)

In [ ]:
new_input = pd.DataFrame({'meal_time_encoded': [le.transform(['dinner'])[0]]})

pred = clf.predict(new_input)
recommended_foods = [food for food, present in zip(y.columns, pred[0]) if present == 1]

print("Recommended foods for dinner:", recommended_foods)

In [37]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# sdf = pd.DataFrame({
#     'meal_time': ['breakfast', 'lunch', 'dinner'],
#     'egg': [2, 1, 1],
#     'tea': [1, 0, 1],
#     'rice': [1, 1, 1],
#     'yam': [1, 1, 1],
#     'chicken': [0, 1, 0],
#     'bread': [2, 0, 1],
#     'beans': [0, 2, 0],
#     'fish': [0, 0, 1]
# })

le = LabelEncoder()
sdf['meal_time_encoded'] = le.fit_transform(sdf['meal_time'])

X = sdf[['meal_time_encoded']]
y = sdf.drop(columns=['meal_time', 'meal_time_encoded'])
y_binary = (y > 0).astype(int)

clf = OneVsRestClassifier(MultinomialNB())
clf.fit(X, y_binary)

y_pred = clf.predict(X)

print(classification_report(y_binary, y_pred, target_names=y.columns, zero_division=0))

# Predict recommendations for lunch
new_input = pd.DataFrame({'meal_time_encoded': [le.transform(['lunch'])[0]]})
pred = clf.predict(new_input)
recommended_foods = [food for food, present in zip(y.columns, pred[0]) if present == 1]
print("Recommended foods for lunch:", recommended_foods)


                    precision    recall  f1-score   support

    diary and milk       0.00      0.00      0.00         0
shawarma and chips       0.00      0.00      0.00         1
            starch       0.00      0.00      0.00         1
     peanut butter       0.00      0.00      0.00         0
              meat       1.00      1.00      1.00         3
           moi-moi       0.67      1.00      0.80         2
        white rice       1.00      1.00      1.00         3
          sharwama       0.00      0.00      0.00         0
           coochie       0.00      0.00      0.00         1
               pap       0.00      0.00      0.00         0
   indomie and egg       0.00      0.00      0.00         1
         groundnut       0.00      0.00      0.00         0
              semo       0.67      1.00      0.80         2
          plantain       1.00      1.00      1.00         3
           burgers       0.00      0.00      0.00         0
               eba       0.67      1.00

/opt/anaconda3/lib/python3.11/site-packages/sklearn/multiclass.py:77: UserWarning: Label not 0 is present in all training examples.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/multiclass.py:77: UserWarning: Label not 3 is present in all training examples.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/multiclass.py:77: UserWarning: Label 4 is present in all training examples.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/multiclass.py:77: UserWarning: Label 6 is present in all training examples.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/multiclass.py:77: UserWarning: Label not 7 is present in all training examples.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/multiclass.py:77: UserWarning: Label not 9 is present in all training examples.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/multiclass.py:77: UserWarning: Label not 11 is present in all 

In [43]:
def get_food_freq_sorted(sdf, meal_time):
    # Select the row for the given meal_time
    row = sdf[sdf['meal_time'] == meal_time]
    if row.empty:
        return f"No data for meal_time: {meal_time}"

    # Drop the meal_time column to keep only food columns
    food_counts = row.drop(columns=['meal_time']).iloc[0]

    # Sort ascending by frequency
    sorted_foods = food_counts.sort_values(ascending=False)

    return sorted_foods


In [78]:
result = get_food_freq_sorted(sdf, 'lunch')
print(result)

jollof rice           73
chicken               66
pounded yam           65
fried rice            63
fish                  61
white rice            61
plantain              58
spaghetti             57
beans                 57
meat                  53
amala                 52
egg                   46
yam                   43
semo                  40
youghurt              30
bread                 27
soda drink            26
meal_time_encoded      2
pizza                  1
eba                    1
fufu                   0
indomie and egg        0
seafood (fish)         0
egusi                  0
fruit salad            0
groundnut              0
pap                    0
snails                 0
coochie                0
sharwama               0
moi-moi                0
peanut butter          0
starch                 0
cereal                 0
burgers                0
okro                   0
salad & fruits         0
tea                    0
sushi                  0
soy                    0


In [57]:
# Assume sdf has columns ['meal_time', 'egg', 'tea', 'rice', ...] with counts per meal_time

def recommend_by_content(sdf, meal_time, top_n=15):
    row = sdf[sdf['meal_time'] == meal_time]
    if row.empty:
        return f"No data for meal_time: {meal_time}"

    food_counts = row.drop(columns=['meal_time']).iloc[0]

    # Sort foods by descending frequency (most frequent foods first)
    recommended = food_counts.sort_values(ascending=False)

    return recommended.head(top_n)

# Usage:
print(recommend_by_content(sdf, 'breakfast'))


egg            99
bread          85
plantain       83
jollof rice    55
chicken        54
Name: 0, dtype: int64


In [64]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Prepare the food matrix: rows=meal_times, columns=foods, values=counts
food_matrix = sdf.set_index('meal_time')

# Compute item-item similarity matrix (foods similarity)
item_sim = cosine_similarity(food_matrix.T)
item_sim_df = pd.DataFrame(item_sim, index=food_matrix.columns, columns=food_matrix.columns)

def recommend_by_collab(meal_time, food_name, top_n=15):
    if food_name not in item_sim_df.columns:
        return f"Food '{food_name}' not found."

    # Get similar foods to the input food
    sim_scores = item_sim_df[food_name].sort_values(ascending=False)

    # Exclude the input food itself
    sim_scores = sim_scores.drop(food_name)

    # Top N similar foods
    top_foods = sim_scores.head(top_n)
    return top_foods

# Example usage:
print(recommend_by_collab('breakfast', 'egg'))


bread          0.989483
youghurt       0.988917
plantain       0.984370
meat           0.919497
spaghetti      0.918125
white rice     0.917755
fish           0.912587
chicken        0.903853
jollof rice    0.898342
yam            0.872069
moi-moi        0.870445
fufu           0.855794
fruit salad    0.843784
tea            0.843784
omlette        0.843784
Name: egg, dtype: float64


In [82]:
def hybrid_recommendation(sdf, meal_time, liked_food, top_n=15):
    # Get top foods in meal_time (should be Series)
    top_foods = recommend_by_content(sdf, meal_time, top_n=20)
    if isinstance(top_foods, str):
        return top_foods  # No data message from recommend_by_content

    # Make sure top_foods is a Series with index = food names
    if not isinstance(top_foods, pd.Series):
        top_foods = pd.Series(top_foods)

    # Get foods similar to liked_food (should be Series)
    similar_foods = recommend_by_collab(meal_time, liked_food, top_n=top_n*2)
    if isinstance(similar_foods, str):
        return similar_foods  # No data message from recommend_by_collab

    # Make sure similar_foods is a Series with index = food names
    if not isinstance(similar_foods, pd.Series):
        similar_foods = pd.Series(similar_foods)

    # Debug prints to verify types and indices
    print(f"Top foods index: {top_foods.index}")
    print(f"Similar foods index: {similar_foods.index}")

    # Filter similar foods to those popular in meal_time
    filtered = similar_foods[similar_foods.index.isin(top_foods.index)].head(top_n)

    # Convert to JSON-friendly format (list of dicts with food and score)
    recommendations = [{"food": food, "score": float(score)} for food, score in filtered.items()]

    response = {
        "meal_time": meal_time,
        "liked_food": liked_food,
        "recommendations": recommendations
    }

    return response

# Example call
print(hybrid_recommendation(sdf, 'lunch', 'white rice'))

Top foods index: Index(['jollof rice', 'chicken', 'pounded yam', 'fried rice', 'fish',
       'white rice', 'plantain', 'spaghetti', 'beans', 'meat', 'amala', 'egg',
       'yam', 'semo', 'youghurt', 'bread', 'soda drink', 'meal_time_encoded',
       'pizza', 'eba'],
      dtype='object')
Similar foods index: Index(['meat', 'chicken', 'jollof rice', 'fish', 'spaghetti', 'fried rice',
       'plantain', 'beans', 'youghurt', 'soda drink', 'egg', 'yam', 'bread',
       'amala', 'pounded yam', 'semo', 'pizza', 'meal_time_encoded', 'moi-moi',
       'fufu', 'fruit salad', 'omlette', 'toast bread', 'coffee', 'cereal',
       'tea', 'eba', 'smoothie', 'egusi', 'coochie'],
      dtype='object')
{'meal_time': 'lunch', 'liked_food': 'white rice', 'recommendations': [{'food': 'meat', 'score': 0.9989088706388158}, {'food': 'chicken', 'score': 0.9978977659204661}, {'food': 'jollof rice', 'score': 0.9975830684933822}, {'food': 'fish', 'score': 0.997534382387587}, {'food': 'spaghetti', 'score': 0.996